# 02 - K Tetangga Terdekat (KNN)
Materi Modul 2: KNN untuk klasifikasi, batas keputusan, pengukuran jarak, pro dan kontra, penskalaan fitur, demo lab.

Ide: sebuah titik baru diberi kelas **mayoritas dari K tetangga terdekatnya**. KNN tidak "belajar" parameter, hanya menyimpan data (lazy learner).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier

wine = pd.read_csv('../datasets/wine.csv')
X, y = wine.drop(columns='target'), wine['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

## 1. KNN dasar

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print('Akurasi test (tanpa scaling):', round(knn.score(X_test, y_test), 4))

## 2. Penskalaan fitur (sangat penting untuk KNN)
KNN memakai jarak, jadi fitur berskala besar (misal `proline` ~ 1000) mendominasi fitur kecil (misal `hue` ~ 1).

In [ ]:
print(X_train[['proline', 'hue', 'alcohol']].describe().loc[['mean', 'std']].round(2))

knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn_scaled.fit(X_train, y_train)
print()
print('Akurasi test TANPA scaling:', round(knn.score(X_test, y_test), 4))
print('Akurasi test DENGAN scaling:', round(knn_scaled.score(X_test, y_test), 4))

## 3. Memilih nilai K
K kecil -> model rumit (overfit). K besar -> model terlalu halus (underfit).

In [ ]:
k_list = range(1, 31)
acc_train, acc_cv = [], []
for k in k_list:
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    m.fit(X_train, y_train)
    acc_train.append(m.score(X_train, y_train))
    acc_cv.append(cross_val_score(m, X_train, y_train, cv=5).mean())

plt.plot(k_list, acc_train, label='Akurasi train')
plt.plot(k_list, acc_cv, label='Akurasi CV')
plt.xlabel('K'); plt.ylabel('Akurasi'); plt.legend(); plt.title('Memilih K')
plt.show()
print('K terbaik menurut CV:', list(k_list)[int(np.argmax(acc_cv))])

## 4. Ukuran jarak

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 6, 8])
print('Euclidean :', round(np.sqrt(np.sum((a - b) ** 2)), 3))
print('Manhattan :', np.sum(np.abs(a - b)))
print('Chebyshev :', np.max(np.abs(a - b)))

In [ ]:
for metrik in ['euclidean', 'manhattan', 'chebyshev']:
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5, metric=metrik))
    skor = cross_val_score(m, X_train, y_train, cv=5).mean()
    print(f'{metrik:10s} | akurasi CV = {skor:.4f}')

## 5. Batas keputusan (2 fitur saja agar bisa digambar)

In [ ]:
def plot_batas(model, X, y, judul=''):
    # menggambar batas keputusan untuk data 2 fitur
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, zz, alpha=0.3, cmap='coolwarm')
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=25)
    plt.title(judul)

In [ ]:
X2 = wine[['alcohol', 'flavanoids']].values
X2 = StandardScaler().fit_transform(X2)
y2 = wine['target'].values

fig = plt.figure(figsize=(15, 4))
for i, k in enumerate([1, 15, 60], start=1):
    plt.subplot(1, 3, i)
    plot_batas(KNeighborsClassifier(n_neighbors=k).fit(X2, y2), X2, y2, f'K = {k}')
plt.tight_layout(); plt.show()

## 6. Bobot tetangga: `uniform` vs `distance`

In [ ]:
for w in ['uniform', 'distance']:
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=10, weights=w))
    print(f'weights={w:9s} | akurasi CV = {cross_val_score(m, X_train, y_train, cv=5).mean():.4f}')

## Pro dan kontra KNN
| Pro | Kontra |
|---|---|
| Sederhana, mudah dipahami | Prediksi lambat untuk data besar |
| Tidak ada proses training | Sensitif terhadap skala fitur |
| Bisa multi-kelas langsung | Kurang baik di dimensi tinggi |

## Catatan
- (cara kerja KNN, kenapa scaling wajib, efek K, dll.)